# Jalon 4 · Validation du plan et des économies

Le notebook contrôle les plans figés dans `resultats_verifies/` et reconstruit la référence sur les mêmes lignes pour chacun des deux périmètres. Les économies sont calculées avec la grille de transport. Le projet ne fournit ni factures historiques ni observations après déploiement.

Les deux scénarios utilisent des populations différentes. Dans le périmètre sans imputation, les charges ou distances inconnues restent exclues du chiffrage.

## Notre méthode, du besoin au camion

1. **Fixer le périmètre.** On travaille par couple fournisseur × usine. Le poids et le volume sont les totaux de la ligne et chaque ligne reste entière. Le scénario avec imputation reprend le jeu préparé. Le scénario sans imputation chiffrable garde les lignes dont le poids, le volume et la distance sont connus.
2. **Ouvrir les dates admissibles.** Une ligne peut partir un jour ouvré de J−7 à J, où J est sa date de besoin. L’horizon commence le 2 novembre 2026. Le moteur examine les 31 profils non vides de jours ouvrés. Une semaine témoin doit contenir un départ pour chacun des jours du profil. L’exception de démarrage reste autorisée pour les besoins qui précèdent le premier départ régulier.
3. **Choisir ensemble les dates et les chargements.** Le solveur affecte chaque ligne à un camion unique, avec un maximum de 24 500 kg et 110,4 m³ par camion. Il minimise le coût total calculé par tranche de poids et de distance. À 126 km, par exemple, les tarifs de base pour 50 kg et 1 000 kg totalisent 49,27 €, contre 51,20 € pour les 1 050 kg regroupés. Le même multiplicateur s’applique aux deux options.
4. **Conserver les plans vérifiés.** Le moteur représente directement les camions et simplifie les combinaisons tarifaires lorsqu’il peut prouver qu’une fusion coûte autant ou moins. Pour les journées les plus denses, il peut calculer une borne avec des capacités agrégées. Les chargements obtenus à partir de ces dates sont ensuite reconstruits et vérifiés camion par camion. Les couples restent indépendants dans le modèle, ce qui permet de répartir les profils et les couples sur plusieurs processus.
5. **Mesurer le gain et la marge restante.** On compare chaque plan à la référence reconstruite sur ses propres lignes. Le coût du meilleur plan faisable fixe le haut de l’intervalle. La borne inférieure fixe le bas. Un couple est certifié lorsque les deux coïncident. Au niveau global, l’écart borne l’économie supplémentaire possible sous les règles modélisées.

Les trois exports donnent les profils et les coûts par couple, les camions avec leurs charges et les affectations des lignes avec leur usine et leur quai.

In [1]:
from collections import defaultdict
from decimal import Decimal
from io import BytesIO
import os
from pathlib import Path
import subprocess
import sys

import numpy as np
import pandas as pd

root = Path.cwd()
if root.name == "milestone4":
    root = root.parent
assert (root / "milestone4" / "resultats_verifies").is_dir(), "Ouvrir le notebook depuis le dépôt ou le dossier milestone4."
sys.path.insert(0, str(root / "src"))

from delivery_schedule_optimization.baseline import BaselineConfig, shipment_cost
from delivery_schedule_optimization.fast_plan import Line, pack
from delivery_schedule_optimization.profile_optimizer import german_national_holidays

config = BaselineConfig()
folder = root / "milestone4" / "resultats_verifies"
pair = ["scope", "supplier_id", "plant"]
truck_id = pair + ["date", "truck"]
source_key = ["supplier_id", "plant", "date", "material", "unloading_point"]

def read_source(path):
    git = ["git", "-C", str(root), "-c", f"safe.directory={root.as_posix()}", "show", f"a0e8aba:{path}"]
    return pd.read_csv(BytesIO(subprocess.check_output(git)))

loads = pd.read_csv(folder / "affectations.csv", parse_dates=["date", "besoin"])
trucks = pd.read_csv(folder / "camions.csv", parse_dates=["date"])
couples = pd.read_csv(folder / "couples.csv", dtype={"profile": str})
print(loads.groupby("scope").agg(lignes=("prepared_id", "size"), usines=("plant", "nunique")).to_string())

                           lignes  usines
scope                                    
avec_imputation              4389       4
sans_imputation_tarifable    4251       4


## Tests, couverture et contraintes

Les tests unitaires couvrent entre autres les seuils tarifaires, le volume, le calendrier et les preuves incomplètes. Le contrôle des exports vérifie la ligne d’origine, les quantités, le poids, le volume, le quai, les dates autorisées et la semaine témoin du profil. L’usine et le quai restent disponibles pour la traçabilité.

In [2]:
env = {**os.environ, "PYTHONPATH": str(root / "src")}
test_run = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "milestone4", "-p", "test_partition_hall.py"],
    cwd=root, env=env, text=True, capture_output=True, check=True,
)
print(test_run.stderr)

prepared = read_source("data/consolidated_dataset_max.csv").reset_index(names="prepared_id")
raw = read_source("docs/released_schedule.csv")
unknown = raw[["weight", "volume"]].fillna(0).le(0).any(axis=1)
distances = read_source("docs/distances.csv")
known_distance = set(distances.loc[distances.distance.gt(0), "supplier_id"])
source = prepared.merge(raw[source_key].assign(charge_inconnue=unknown), on=source_key, validate="1:1")
priced = ~source.charge_inconnue & source.supplier_id.isin(known_distance)
expected = {"avec_imputation": set(source.prepared_id),
            "sans_imputation_tarifable": set(source.loc[priced, "prepared_id"])}
assert not loads.duplicated(["scope", "prepared_id"]).any()
assert {scope: set(g.prepared_id) for scope, g in loads.groupby("scope")} == expected
assert set(trucks[truck_id].itertuples(index=False, name=None)) == set(loads[truck_id].itertuples(index=False, name=None))

aligned = loads.merge(source.rename(columns={"date": "besoin"}), on="prepared_id", suffixes=("", "_pinned"), validate="m:1")
for col in ["supplier_id", "plant", "material", "unloading_point"]:
    assert aligned[col].eq(aligned[f"{col}_pinned"]).all(), col
assert aligned.besoin.eq(pd.to_datetime(aligned.besoin_pinned)).all()
for col in ["schedule_quantity", "weight", "volume", "distance"]:
    np.testing.assert_allclose(aligned[col], aligned[f"{col}_pinned"], rtol=0, atol=1e-8)

advance = (loads.besoin - loads.date).dt.days
holidays = german_national_holidays(set(loads.date.dt.year))
assert advance.between(0, 7).all() and loads.date.ge(pd.Timestamp("2026-11-02")).all()
assert loads.date.dt.dayofweek.lt(5).all() and not loads.date.dt.date.isin(holidays).any()
anchor = loads.besoin.min()
workdays = pd.date_range(anchor, loads.besoin.max(), freq="B")
workdays = workdays[~pd.Index(workdays.date).isin(holidays)]
for _, group in loads.groupby(pair):
    profile = set(map(int, str(group.profile.iloc[0]).split()))
    assert group.profile.nunique() == 1
    regular = group.date.dt.dayofweek.isin(profile)
    first = min((day for day in workdays if day.weekday() in profile), default=None)
    witness = group.loc[regular].groupby(group.date.dt.to_period("W-SUN")).date.agg(lambda d: set(d.dt.dayofweek))
    assert any(profile == covered for covered in witness)
    assert group.loc[~regular, "date"].ge(anchor).all()
    if first is not None:
        assert group.loc[~regular, "besoin"].lt(first).all()

audited = loads.groupby(truck_id, as_index=False).agg(
    lines=("prepared_id", "size"), weight=("weight", "sum"), volume=("volume", "sum"),
    distance=("distance", "first"), distances=("distance", "nunique"),
)
paired = audited.merge(trucks, on=truck_id, suffixes=("", "_export"), validate="1:1")
assert paired.distances.eq(1).all()
assert paired.weight.le(config.truck_kg + 1e-8).all() and paired.volume.le(config.truck_m3 + 1e-8).all()
for col in ["lines", "weight", "volume"]:
    np.testing.assert_allclose(paired[col], paired[f"{col}_export"], rtol=0, atol=1e-8)
print(f"{len(loads):,} affectations et {len(trucks):,} camions vérifiés dans les deux périmètres.")
print(f"{(~priced).sum()} exclusions du scénario sans imputation chiffrable.")

...........
----------------------------------------------------------------------
Ran 11 tests in 0.274s

OK

8,640 affectations et 1,383 camions vérifiés dans les deux périmètres.
138 exclusions du scénario sans imputation chiffrable.


## Avant / après sur un périmètre identique

La référence groupe les lignes à leur date de besoin, puis les charge avec la méthode de référence du moteur. Chaque camion est tarifé séparément. On rapproche ensuite ces montants des résultats figés par couple.

In [3]:
grid = read_source("docs/rate_sheet.csv").set_index("km_up_to")

def threshold(value):
    return np.inf if str(value).startswith(">") else float(value)

grid.index, grid.columns = grid.index.map(threshold), grid.columns.map(threshold)
grid = grid.sort_index().sort_index(axis=1)

def price(frame):
    return shipment_cost(grid, frame.distance, frame.weight, np.ones(len(frame)), config)

np.testing.assert_allclose(price(paired), paired.cost_eur, rtol=0, atol=1e-6)

baseline_rows = []
for key, group in loads.groupby(pair, sort=False):
    needs = defaultdict(list)
    for r in group.itertuples(index=False):
        due = r.besoin.date()
        needs[due].append(Line(str(r.prepared_id), r.supplier_id, r.plant, r.unloading_point,
                               r.material, due, r.schedule_quantity, r.weight, r.volume, r.distance))
    for truck in pack(needs):
        weight = sum(Decimal(str(line.kg)) for line in truck.lines)
        baseline_rows.append(dict(zip(pair, key), date=pd.Timestamp(truck.day), weight=float(weight),
                                  volume=truck.m3, distance=truck.lines[0].km))
before = pd.DataFrame(baseline_rows)
before["cost_eur"] = price(before)
after = trucks.copy()

per_pair = (
    before.groupby(pair).agg(cout_avant=("cost_eur", "sum"), camions_avant=("date", "size"))
    .join(after.groupby(pair).agg(cout_apres=("cost_eur", "sum"), camions_apres=("date", "size")))
    .join(couples.set_index(pair)[["reference", "upper", "lower", "certified"]])
)
factor = config.price_factor / 100
np.testing.assert_allclose(per_pair.cout_avant, per_pair.reference * factor, rtol=0, atol=1e-6)
np.testing.assert_allclose(per_pair.cout_apres, per_pair.upper * factor, rtol=0, atol=1e-6)
per_pair["gain_calcule"] = per_pair.cout_avant - per_pair.cout_apres
per_pair["marge_max"] = (per_pair.upper - per_pair.lower) * factor
overview = per_pair[["cout_avant", "cout_apres", "camions_avant", "camions_apres", "gain_calcule", "marge_max"]].groupby("scope").sum()
overview["gain_pct"] = 100 * overview.gain_calcule / overview.cout_avant
print(overview.round(2).to_string())

                           cout_avant  cout_apres  camions_avant  camions_apres  gain_calcule  marge_max  gain_pct
scope                                                                                                             
avec_imputation             176724.61   109345.78           1741            756      67378.83    1677.09     38.13
sans_imputation_tarifable   142964.05    75895.85           1603            627      67068.20    1498.05     46.91


## Distributions avant / après

La médiane et les quantiles sont calculés par camion. Les taux de remplissage utilisent la capacité maximale en poids et en volume. La dispersion des coûts tient compte du tarif unitaire de chaque camion.

In [4]:
def distribution(frame, stage):
    return frame.assign(etape=stage, poids_pct=100 * frame.weight / config.truck_kg,
                        volume_pct=100 * frame.volume / config.truck_m3)

transports = pd.concat([distribution(before, "avant"), distribution(after, "après")], ignore_index=True)
quantiles = transports.groupby(["scope", "etape"])[["poids_pct", "volume_pct", "cost_eur"]].quantile([.25, .5, .75, .9])
quantiles.index.names = ["scope", "etape", "quantile"]
print(quantiles.round(2).to_string())

daily = transports.groupby(["scope", "etape", "date"]).size()
print()
print("Pic de camions par jour")
print(daily.groupby(["scope", "etape"]).max().unstack().to_string())

                                          poids_pct  volume_pct  cost_eur
scope                     etape quantile                                 
avec_imputation           après 0.25           1.14        0.80     64.78
                                0.50           4.07        2.28    125.39
                                0.75          29.83        7.85    213.92
                                0.90         100.00      100.00    295.92
                          avant 0.25           0.58        0.39     52.95
                                0.50           1.49        1.09     74.67
                                0.75           4.48        2.17    132.20
                                0.90          24.56        6.38    213.92
sans_imputation_tarifable après 0.25           0.79        0.75     53.24
                                0.50           2.74        1.61    101.12
                                0.75           9.60        4.53    171.86
                                0.90  

## Contributions et limite du gain potentiel

La contribution d’une usine rapporte son gain calculé au gain total du même périmètre. Le registre détaille aussi le classement des couples. La marge maximale vient de la borne inférieure de coût. Elle indique le plafond d’un gain supplémentaire sous les règles modélisées.

In [5]:
by_plant = per_pair[["cout_avant", "cout_apres", "gain_calcule", "marge_max"]].groupby(["scope", "plant"]).sum()
by_plant["part_du_gain_pct"] = 100 * by_plant.gain_calcule / by_plant.groupby("scope").gain_calcule.transform("sum")
print(by_plant.round(2).to_string())

top = per_pair.sort_values("gain_calcule", ascending=False).groupby("scope").head(5)
print()
print("Cinq plus fortes contributions dans chaque périmètre")
print(top[["gain_calcule", "marge_max", "certified"]].round(2).to_string())
print()
print("Hausse de coût face à la référence par périmètre")
print(per_pair.gain_calcule.lt(-.005).groupby("scope").sum().to_string())

                                   cout_avant  cout_apres  gain_calcule  marge_max  part_du_gain_pct
scope                     plant                                                                     
avec_imputation           PLT_001    45200.68    27611.06      17589.62     373.70             26.11
                          PLT_002    45004.43    28491.84      16512.59     291.42             24.51
                          PLT_003    27931.81    19232.51       8699.30       0.00             12.91
                          PLT_004    58587.68    34010.37      24577.31    1011.96             36.48
sans_imputation_tarifable PLT_001    38666.83    21109.38      17557.45     360.41             26.18
                          PLT_002    33572.69    17199.83      16372.86     125.68             24.41
                          PLT_003    19595.85    11035.27       8560.58       0.00             12.76
                          PLT_004    51128.69    26551.37      24577.31    1011.96         